<a href="https://colab.research.google.com/github/akshat280706/ML-Lab-Experiment/blob/main/testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#ex2 entropy/info gain
import pandas as pd
import math
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import(accuracy_score, confusion_matrix, precision_score,recall_score, f1_score)
data=pd.read_csv("mushrooms.csv")
target="class"

def entropy(data):
  counts=data[target].value_counts()
  total=len(data)
  e=0
  for count in counts:
    p=count/total
    e-=p*math.log2(p)
  return e

def information_gain(data, attribute):
  total_entropy=entropy(data)
  weighted_entropy=0

  for value in data[attribute].unique():
    subset=data[data[attribute]==value]
    weight=len(subset)/len(data)
    weighted_entropy+=weight*entropy(subset)
  return total_entropy-weighted_entropy


def ID3(data,attributes):
  classes=data[target]

  if len(classes.unique())==1:
    return classes.iloc[0]
  if len(attributes)==0:
    return classes.mode()[0]

  print("\ncurrent entropy: ", round(entropy(data),5))
  print("\ninformation gain: ")

  best_attribute=None
  best_gain=-1

  for attribute in attributes:
    current_gain=information_gain(data,attribute)
    print(attribute,"=",round(current_gain,5))
    if(current_gain>best_gain):
      best_gain=current_gain
      best_attribute=attribute
  print("selected attribute: ", best_attribute)
  tree={best_attribute:{}}

  remaining_attribute=[
      attribute
      for attribute in attributes
      if attribute!=best_attribute
  ]

  for value in data[best_attribute].unique():
    subset=data[data[best_attribute]==value]

    if len(subset)==0:
      tree[best_attribute][value]=classes.mode()[0]
    else:
      subset=subset.drop(columns=[best_attribute])

      tree[best_attribute][value]=ID3(subset, remaining_attribute)
  return tree

def predict(tree, row):
    if not isinstance(tree, dict):
        return tree

    root = list(tree.keys())[0]
    value = row[root]

    if value not in tree[root]:
        return None
    return predict(tree[root][value], row)

def print_rules(tree, rule=""):
  if not isinstance(tree, dict):
    print(rule,"then class= ", tree)
    return
  root=list(tree.keys())[0]
  for value in tree[root]:
    if rule=="":
      new_rule="if "+root+" = "+str(value)
    else:
      new_rule=rule+" and "+root+" = "+str(value)
    print_rules(tree[root][value],new_rule)


train_data, test_data = train_test_split(data,test_size=0.2,random_state=42,stratify=data[target])
attributes = list(train_data.columns)
attributes.remove(target)

tree = ID3(train_data, attributes)
print("\nRoot node: ")
print(list(tree.keys())[0])
print("\nTree generation rules: ")
print_rules(tree)

predictions = []
for _, row in test_data.iterrows():
    predictions.append(predict(tree, row))

correct=sum(
    actual == predicted
    for actual, predicted in zip(test_data[target],predictions)
)

manual_accuracy = correct/len(test_data)
print("\nID3 accuracy(manual):")
print(round(manual_accuracy * 100, 2), "%")


encoded=data.copy()

for column in encoded.columns:
    encoded[column]=LabelEncoder().fit_transform(encoded[column])

X = encoded.drop(target, axis=1)
Y = encoded[target]

X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,random_state=42,stratify=Y)
model=DecisionTreeClassifier(criterion="entropy",random_state=42)
model.fit(X_train, Y_train)
prediction = model.predict(X_test)
print("\nScikit-learn accuracy:")
print(round(accuracy_score(Y_test, prediction) * 100, 2), "%")
print("\nConfusion Matrix")
print(confusion_matrix(Y_test, prediction))
print("\nPrecision")
print(precision_score(Y_test, prediction))
print("\nRecall")
print(recall_score(Y_test, prediction))
print("\nF1 Score")
print(f1_score(Y_test, prediction))
print("\nComparison")
if manual_accuracy > accuracy_score(Y_test, prediction):
    print("manual code better")
elif manual_accuracy < accuracy_score(Y_test, prediction):
    print("Scikit-learn is better.")
else:
    print("same results.")


FileNotFoundError: [Errno 2] No such file or directory: 'mushrooms.csv'

In [ ]:
#3infogain, entropy
import pandas as pd
import math
from sklearn.model_selection import train_test_split

data=pd.read_csv("mushrooms.csv")
target="class"
train_data,test_data=train_test_split(data,test_size=0.2,random_state=42,stratify=data[target])
from sklearn.metrics import (confusion_matrix,precision_score,recall_score,f1_score)

def entropy(data):
  counts=data[target].value_counts()
  total=len(data)
  e=0
  for count in counts:
    p=count/total
    e-=p*math.log2(p)
  return e


def information_gain(data, attribute):
  total_entropy=entropy(data)
  weighted_entropy=0

  for value in data[attribute].unique():
    subset=data[data[attribute]==value]
    weight=len(subset)/len(data)
    weighted_entropy+=weight*entropy(subset)
  return total_entropy-weighted_entropy

def gain_ratio(data,attribute):
  gain=information_gain(data,attribute)
  split_info=0

  for value in data[attribute].unique():
    subset=data[data[attribute]==value]
    p=len(subset)/len(data)

    split_info-=p*math.log2(p)

  if split_info==0:
    return 0
  return gain/split_info


def gini(data):
  counts=data[target].value_counts()
  g=1

  for count in counts:
    p=count/len(data)
    g-=p**2
  return g

def gini_gain(data,attribute):
  total=0
  for value in data[attribute].unique():
    subset=data[data[attribute]==value]
    weight=len(subset)/len(data)
    total+=weight*gini(subset)
  return total

def misclassification(data):
  counts=data[target].value_counts()
  maxi=max(counts)
  return 1-(maxi/len(data))

def misclassification_gain(data,attribute):
  total=0
  for value in data[attribute].unique():
    subset=data[data[attribute]==value]
    weight=len(subset)/len(data)
    total+=weight*misclassification(subset)
  return total

def best_attribute(data, attributes, method):
    scores = {}
    print("\nScores")

    for attribute in attributes:
        if method == "gain":
            score = information_gain(data, attribute)
        elif method == "ratio":
            score = gain_ratio(data, attribute)
        elif method == "gini":
            score = -gini_gain(data, attribute)
        else:
            score = -misclassification_gain(data, attribute)

        scores[attribute] = score
        print(attribute, "=", round(score, 5))
    return max(scores, key=scores.get)

def create_tree(data,attributes,methods):
  classes=data[target]
  if len(classes.unique())==1:
    return classes.iloc[0]
  if len(attributes)==0:
    return classes.mode()[0]

  best = best_attribute(data, attributes, method)
  print("\nselected:", best)
  tree = {best: {}}

  remaining_attribute = attributes.copy()
  remaining_attribute.remove(best)

  for value in data[best].unique():
    subset=data[data[best]==value]

    if len(subset)==0:
      tree[best][value]=classes.mode()[0]
    else:
      tree[best][value]=create_tree(subset,remaining_attribute,method)
  return tree

def predict(tree, row):
    if not isinstance(tree, dict):
        return tree

    root = list(tree.keys())[0]
    value = row[root]

    if value not in tree[root]:
        return None
    return predict(tree[root][value], row)


def print_rules(tree, rule=""):
  if not isinstance(tree, dict):
    print(rule,"then class= ", tree)
    return
  root=list(tree.keys())[0]
  for value in tree[root]:
    if rule=="":
      new_rule="if "+root+" = "+str(value)
    else:
      new_rule=rule+" and "+root+" = "+str(value)
    print_rules(tree[root][value],new_rule)


attributes = list(train_data.columns)
attributes.remove(target)
methods = {
    "gain": "Information Gain",
    "ratio": "Gain Ratio",
    "gini": "Gini Index",
    "misclassification": "Misclassification Error"
}

for method in methods:
    print(methods[method])

    tree = create_tree(train_data,attributes,method)



    print("\nroot node")
    print(list(tree.keys())[0])

    print("\nFollowing rules: ")
    print_rules(tree)

    predictions = []

    for _, row in test_data.iterrows():
        predictions.append(predict(tree, row))

    correct = sum(
        actual == predicted
        for actual, predicted in zip(
            test_data[target],
            predictions
        )
    )
    accuracy = correct / len(test_data)
    print("\nAccuracy =", round(accuracy * 100, 2), "%")

    actual = test_data[target]
    cm = confusion_matrix(actual, predictions)
    print("\nConfusion Matrix: ")
    print(cm)

In [ ]:
#4 naive bayes
import pandas as pd
import numpy as np
import math
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import CategoricalNB

data=pd.read_csv("mushrooms.csv")
target="class"

#encoding
encoders={}
for column in data.columns:
  encoder=LabelEncoder()
  data[column]=encoder.fit_transform(data[column])
  encoders[column]=encoder

#input
X=data.drop("class",axis=1)
#output
y=data["class"]

#train and test
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42, stratify=y)
print("training set size: ", X_train.shape)
print("testing set size: ", X_test.shape)

#unique classes
classes = np.unique(y_train)
print("\nClasses:")
print(classes)

#prior probability
priors = {}
for cls in classes:
    class_data = X_train[y_train== cls]
    priors[cls] = len(class_data) / len(X_train)
print("\nPrior Probabilities: ")
for cls in priors:
  print("prob(class=",cls,"): ", priors[cls])

#likelihood prob
likelihoods={}
for cls in classes:
  likelihoods[cls]={}
  class_data=X_train[y_train==cls]
  #now going through each feature
  for column in X_train.columns:
    likelihoods[cls][column]={}
    unique_values=X_train[column].unique()
    total=len(class_data)
    #value of current feature
    for value in unique_values:
      count=np.sum(class_data[column]==value)

      #laplace smoothing
      probability=(count+1)/(total+len(unique_values))
      # probability=count/total

      #storing the probability now
      likelihoods[cls][column][value]=probability

#printing likelihood calculations
print("\n")
print("likelihood calculations: ")
feature=X_train.columns[0]
print("Features: ", feature)

for cls in classes:
  print("\nClass=",cls)
  for value in likelihoods[cls][feature]:
    #retrieving the probability
    probability=likelihoods[cls][feature][value]
    print("Prob(",feature,"=",value,"/class=",cls,"): ",probability)

#predictions
predictions=[]
for index,row in X_test.iterrows():
  probabilities={}
  for cls in classes:
    probability=np.log(priors[cls])

    #likelihood of alll features
    for column in X_test.columns:
      value=row[column]
      probability+=np.log(likelihoods[cls][column][value])
    #storing the probability
    probabilities[cls]=probability
  prediction=max(probabilities,key=probabilities.get)
  predictions.append(prediction)

#converting into array for confusion matrix and acc score etc
predictions=np.array(predictions)

print("\nNaive bayes: ")
#confusion matrix
cm=confusion_matrix(y_test,predictions)
print("confusion matrix: \n",cm)
#accuracy
acc=accuracy_score(y_test,predictions)
print("accuracy score: ",acc)
#precision
pre=precision_score(y_test,predictions)
print("precision score: ",pre)
#recall
rec=recall_score(y_test,predictions)
print("recall score: ",rec)


#library NB
library_model=CategoricalNB()
library_model.fit(X_train,y_train)
library_prediction = library_model.predict(X_test)

print("\n")
print("Library bayes: ")
#confusion matrix
cm=confusion_matrix(y_test,library_prediction)
print("confusion matrix: \n",cm)
#accuracy
acc=accuracy_score(y_test,library_prediction)
print("accuracy score: ",acc)
#precision
pre=precision_score(y_test,library_prediction)
print("precision score: ",pre)
#recall
rec=recall_score(y_test,library_prediction)
print("recall score: ",rec)

In [ ]:
#5linear reg
import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import(mean_squared_error, mean_absolute_error)

df=pd.read_csv("electricity_cost_dataset.csv")

print("shape of the dataset: ")
print(df.shape)
print("\n")

#input
X=df.drop("electricity cost", axis=1)

#output
y=df["electricity cost"]

#encoding

#using oneHotEncoding to convert into 0/1
X=pd.get_dummies(X,columns=["structure type"], drop_first=True)
#drop_first as one true other false

#to store decimal
X=X.astype(float)

#spliting
X_train,X_test,y_train,y_test=train_test_split(X,y, test_size=0.2, random_state=42)

#feature scaling with mean 0, SD 1
scaler=StandardScaler()
#calulate mean and sd and then standerdize
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test) #uses the same mean and SD

#convert into numpy array
y_train=y_train.values
y_test=y_test.values


n_samples=X_train.shape[0] #no of traiing rows
n_features=X_train.shape[1] #col

#how much it contributes
weights=np.zeros(n_features)

bias=0 #constant added to prediction

learning_rate=0.01
iterations=1000 #how much times model willl go through training

error_history=[] #store error after every epoch

#gradient descent
for i in range(iterations):

  #prediction
  predictions=np.dot(X_train, weights)+bias
  error=predictions-y_train

  #mean square error(how bad model is performing)
  mse=np.mean(error**2)
  error_history.append(mse)

  #gradient cal(how much weight should change)
  #taking the derivative of mse
  dw=(2/n_samples)*np.dot(X_train.T,error)

  #bias gradient(how should bias change)
  db=(2/n_samples)*np.sum(error)

  weights=weights-learning_rate*dw
  bias=bias-learning_rate*db


#callculating actual predictions
predictions=np.dot(X_test, weights)+bias

print("\npredicted vals: ", predictions[:10]) #only first 10
print("\nactual vals: ", y_test[:10])


#performance
mse=mean_squared_error(y_test,predictions)
rmse=np.sqrt(mse)
mae=mean_absolute_error(y_test,predictions)

print("\npredictions: ")
print("mse: ",mse)
print("rmse: ",rmse)
print("mae: ",mae)

#graph
plt.figure(figsize=(8, 5))
plt.plot(range(1, iterations + 1),error_history)
plt.xlabel("Epoch")
plt.ylabel("Mean Squared Error")
plt.title("Error vs Epoch")
plt.grid()
plt.show()

#lib
library_model = LinearRegression()

library_model.fit(X_train, y_train)
library_prediction = library_model.predict(X_test)
library_mse = mean_squared_error(y_test,library_prediction)
library_rmse = np.sqrt(library_mse)
library_mae = mean_absolute_error(y_test,library_prediction)

print("\nlib LR")

print("MSE  :", library_mse)
print("RMSE :", library_rmse)
print("MAE  :", library_mae)

In [ ]:
#6logistic reg
import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt
from sklearn.metrics import(confusion_matrix, accuracy_score,
precision_score, recall_score,f1_score,roc_curve, roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

data=pd.read_csv("mushrooms.csv")
print("Dataset shape:", data.shape)
#converting categorial into numeical
X=pd.get_dummies(data.drop("class", axis=1),dtype=float)
#e=0(edible) p=1(poi)
y=data["class"].map({"e":0, "p":1}).values

X=X.values
y=y.astype(float)
#spliting
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2,
                                                    random_state=42,stratify=y)
print("\nTraining data:", X_train.shape)
print("Testing data:", X_test.shape)

#logistic regression

def sigmoid(z):
  return 1/(1+np.exp(-np.clip(z,-500,500)))
#if z smaller than -500, make it -500

def train_logistic(X,y,learning_rate=0.1,epoch=1000):
  rows,columns=X.shape
  weights=np.zeros(columns)
  bias=0

  for i in range(epoch):
    #z=Xw+b
    z=np.dot(X,weights)+bias
    #cal predictions(to conv into prob)
    predictions=sigmoid(z)
    #error
    error=predictions-y

    #gradient(how should each weight change)
    dw=np.dot(X.T, error)/rows
    #mean error
    db=np.mean(error)

    weights=weights-learning_rate*dw #gradient descent update
    bias=bias-learning_rate*db
  return weights,bias

weights,bias=train_logistic(X_train,y_train)

#predictions
def predict_probability(X,weights,bias):
  z=np.dot(X,weights)+bias
  return sigmoid(z) #converts val b/w 0 and 1

def predict(X,weights,bias):
  probabilities=predict_probability(X,weights,bias)
  return (probabilities>=0.5).astype(int) #true=1, false=0
  #>=0.5: class 1 else class 0

#predictions
y_pred=predict(X_train,weights,bias)
def predict_probability(X, weights, bias):

    z = np.dot(X, weights) + bias

    return sigmoid(z)


def predict(X, weights, bias):

    probabilities = predict_probability(X, weights, bias)

    return (probabilities >= 0.5).astype(int)


# Predictions
y_pred = predict(X_test, weights, bias)
y_prob = predict_probability(X_test, weights, bias)

print("\n logistic reg: ")
#confusion matrix
cm=confusion_matrix(y_test,y_pred)
print("confusion matrix: \n",cm)
#accuracy
acc=accuracy_score(y_test,y_pred)
print("accuracy score: ",acc)
#precision
pre=precision_score(y_test,y_pred)
print("precision score: ",pre)
#recall
rec=recall_score(y_test,y_pred)
print("recall score: ",rec)
#f1
print("F1 Score:", f1_score(y_test, y_pred))


#standard library
library_model = LogisticRegression(max_iter=1000)
library_model.fit(X_train, y_train)
library_pred = library_model.predict(X_test)
library_prob = library_model.predict_proba(X_test)[:, 1]
print("\nlibrary: ")
print("Confusion Matrix:", confusion_matrix(y_test, library_pred))
print("Accuracy:", accuracy_score(y_test, library_pred))
print("Precision:", precision_score(y_test, library_pred))
print("Recall:",recall_score(y_test, library_pred))
print("F1 Score:",f1_score(y_test, library_pred))

#roc curve
fpr_manual, tpr_manual, _ = roc_curve(y_test, y_prob)
fpr_library, tpr_library, _ = roc_curve(y_test, library_prob)
plt.figure(figsize=(8, 6))

plt.plot(fpr_manual,tpr_manual,label="Logistic Regression")
plt.plot(fpr_library,tpr_library,label="library")
plt.plot([0, 1],[0, 1],linestyle="--",label="Random Classifier")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.title("ROC Curve")
plt.legend()
plt.grid()

plt.show()

In [ ]:
#7neural nw
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import precision_score, recall_score, f1_score

df=pd.read_csv("mushrooms.csv")
print("datashape: ", df.shape)

X = df.drop("class", axis=1)
y = df["class"]

#encoding
y = y.map({"e": 0,"p": 1}).values
X=pd.get_dummies(X,dtype=float).values.astype(np.float64)
print("encoded dshape:", X.shape)

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42
                                               ,stratify=y)
#representation each into columns
X_train=X_train.T
X_test=X_test.T
y_train = y_train.reshape(1,-1)
y_test = y_test.reshape(1,-1)

from inspect import Parameter
#activationfnxs
def relu(Z):
  return np.maximum(0,Z)
def relu_derivative(Z):
  return (Z>0).astype(float)

def sigmoid(Z):
  Z=np.clip(Z,-500,500)
  return 1/(1+np.exp(-Z))

def initialize_parameters(input_size,hidden1_size,hidden2_size,output_size):
  np.random.seed(42)

  #applying he initialization(root(2/no of i/p))
  w1=np.random.randn(hidden1_size,input_size)*np.sqrt(2/input_size)
  b1=np.zeros((hidden1_size,1))
  w2=np.random.randn(hidden2_size,hidden1_size)*np.sqrt(2/hidden1_size)
  b2=np.zeros((hidden2_size,1))
  w3=np.random.randn(output_size,hidden2_size)*np.sqrt(2/hidden2_size)
  b3=np.zeros((output_size,1))

  return{"w1":w1, "b1":b1,"w2":w2, "b2":b2,"w3":w3, "b3":b3}

#forward propogation
def forward_propogation(X,parameters):
  w1=parameters["w1"]
  b1=parameters["b1"]

  w2=parameters["w2"]
  b2=parameters["b2"]

  w3=parameters["w3"]
  b3=parameters["b3"]

  Z1=np.dot(w1,X)+b1
  A1=relu(Z1)
  Z2=np.dot(w2,A1)+b2
  A2=relu(Z2)
  Z3=np.dot(w3,A2)+b3
  A3=sigmoid(Z3)

  vals = {"Z1":Z1,"A1":A1,"Z2":Z2,"A2":A2,"Z3":Z3,"A3":A3}
  return A3,vals

#loss fxn
#comparing actual y with predicted
def compute_loss(Y,AL):
  m=Y.shape[1] #no. training ex
  AL= np.clip(AL, 1e-8,1-1e-8) #else log(0)
  loss=-(1/m)*np.sum(Y*np.log(AL)+(1-Y)*np.log(1-AL))
  return loss

#backpropogation(how much should each wt and bias change)
def backward_propogation(X,Y,parameters,vars):
  m=X.shape[1]
  w2 = parameters["w2"] #for layer1
  w3 = parameters["w3"] #for layer2
  Z1=vars["Z1"]
  A1=vars["A1"]
  Z2=vars["Z2"]
  A2=vars["A2"]
  A3=vars["A3"]

  #output layer
  dZ3=A3-Y
  dw3=(1/m)*np.dot(dZ3,A2.T)
  db3=(1/m)*np.sum(dZ3,axis=1,keepdims=True)

  #hidden layer2
  dA2=np.dot(w3.T,dZ3)
  dZ2=dA2*relu_derivative(Z2)
  dw2=(1/m)*np.dot(dZ2,A1.T)
  db2=(1/m)*np.sum(dZ2,axis=1,keepdims=True)

  #hidden layer1
  dA1=np.dot(w2.T,dZ2)
  dZ1=dA1*relu_derivative(Z1)
  dw1=(1/m)*np.dot(dZ1,X.T)
  db1=(1/m)*np.sum(dZ1,axis=1,keepdims=True)

  return {"dw1":dw1,"db1":db1,"dw2":dw2,"db2":db2,"dw3":dw3,"db3":db3}

#updating parameters
def update_parameters(parameters,gradients,learning_rate):
  parameters["w1"]-=(learning_rate*gradients["dw1"])
  parameters["b1"]-=(learning_rate*gradients["db1"])
  parameters["w2"]-=(learning_rate*gradients["dw2"])
  parameters["b2"]-=(learning_rate*gradients["db2"])
  parameters["w3"]-=(learning_rate*gradients["dw3"])
  parameters["b3"]-=(learning_rate*gradients["db3"])
  return parameters

def train_neural_network(X,Y,hidden1_size=64,hidden2_size=32,learning_rate=0.01,
                         epochs=500):
  input_size=X.shape[0]
  parameters=initialize_parameters(input_size,hidden1_size,hidden2_size,1)
  losses=[]
  for epoch in range(epochs):
    #forward prop
    AL,vars=forward_propogation(X,parameters)
    #loss
    loss=compute_loss(Y,AL)
    #backward prop
    gradients=backward_propogation(X,Y,parameters,vars)
    #update
    parameters=update_parameters(parameters,gradients,learning_rate)
    losses.append(loss)
    if epoch%100==0: #printing each 100 epochs
      print("epoch: ",epoch,"loss:",round(loss,6))

  return parameters,losses

#calling above fxn
parameters,losses=train_neural_network(X_train,y_train,hidden1_size=64,
                                          hidden2_size=32,learning_rate=0.01,
                                          epochs=500)

#predictions
def predict(X,parameters):
  AL,_=forward_propogation(X,parameters)
  return (AL >= 0.5).astype(int)

y_pred_nn=predict(X_test,parameters)

#accuracy
accuracy_nn=np.mean(y_pred_nn==y_test)*100
print("\naccuracy: ",round(accuracy_nn,2),"%")

#confusion matrix
print("\nconfusion matrix: ")
print(confusion_matrix(y_test.flatten(),y_pred_nn.flatten()))

precision_nn=precision_score(y_test.flatten(),y_pred_nn.flatten())
recall_nn=recall_score(y_test.flatten(),y_pred_nn.flatten())
f1_nn=f1_score(y_test.flatten(),y_pred_nn.flatten())
print("Precision:",round(precision_nn,2))
print("Recall:",round(recall_nn,2))
print("F1Score:",round(f1_nn,2))


#graph
plt.figure(figsize=(8, 5))
plt.plot(losses)
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("curve")
plt.grid(True)
plt.show()


#library
X_train_tree = X_train.T
X_test_tree = X_test.T
y_train_tree = y_train.flatten()
y_test_tree = y_test.flatten()

tree_model = DecisionTreeClassifier(criterion="entropy",random_state=42)
tree_model.fit(X_train_tree,y_train_tree)
y_pred_tree = tree_model.predict(X_test_tree)
accuracy_tree = accuracy_score(y_test_tree,y_pred_tree)*100
print("\nlib acc",round(accuracy_tree, 2),"%")
print("\nlib confusion matrix: ")
print(confusion_matrix(y_test_tree,y_pred_tree))


In [ ]:
#8SVM
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.svm import SVC
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score

df=pd.read_csv("heart_cleveland_upload.csv")

df.columns=df.columns.str.strip().str.lower()
df=df.replace("?",np.nan)
for col in df.columns:
  df[col]=pd.to_numeric(df[col],errors="coerce")
df=df.dropna()
target="condition"
X=df.drop(columns=[target]).values.astype(float)
y=(df[target].values>0).astype(int)
print("shape:", df.shape)
print("without disease: ", np.sum(y==0))
print("with disease: ", np.sum(y==1))

X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,
                                               random_state=42,stratify=y)
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)
y_svm=np.where(y_train==1,1,-1)
print("training: ",len(X_train))
print("testing: ",len(X_test))

def print_metrics(y_test, y_pred, model_name):
    print(f"\n{model_name}")
    print("Accuracy:",accuracy_score(y_test,y_pred))
    print("Precision:",precision_score(y_test,y_pred,zero_division=0))
    print("Recall:",recall_score(y_test,y_pred,zero_division=0))
    print("F1-score:",f1_score(y_test,y_pred,zero_division=0))
    print("\n")

#library linear svm baseline
sklearn_linear_svm=SVC(kernel="linear")
sklearn_linear_svm.fit(X_train,y_train)
y_pred_sklearn_linear=sklearn_linear_svm.predict(X_test)
print_metrics(y_test,y_pred_sklearn_linear,"Library Linear SVM")

#SVM(SGD)
#converting labels
Y1=np.where(y_train<=0,-1,1)
n_samples,n_features=X_train.shape
W=np.zeros(n_features)
B=0.0
alpha=0.001
c=5
epochs=1000
#stochastic grad descent
epoch_errors_linear = []
for epoch in range(epochs):
  for i in range(n_samples):
    #decision score
    score=np.dot(X_train[i],W)+B

    #margin condition
    if Y1[i]*score>=1:
      #satsisfied cond
      W=W-alpha*W
    else:
      #condt not satisfied
      W=W-alpha*(W-c*Y1[i]*X_train[i] b b )
      B=B-alpha*(-c*Y1[i])

  #error after epoch
  train_scores=np.dot(X_train,W)+B
  train_predictions=np.where(train_scores>0,1,0)
  error=np.mean(train_predictions!=y_train)
  epoch_errors_linear.append(error)

#test scores
y_predd=np.dot(X_test,W)+B
y_pred_linear=np.where(y_predd<=0,0,1)
print_metrics(y_test,y_pred_linear,"my svm")
#graph
plt.figure(figsize=(8, 5))
plt.plot(range(1,epochs+1),epoch_errors_linear,color="blue",label="SVM")
plt.xlabel("Epoch")
plt.ylabel("error")
plt.title("graph")
plt.grid(True)
plt.legend()
plt.show()

#support vector
final_scores=np.dot(X_train,W)+B
#training ex on or inside margin
support_vector_mask=Y1*final_scores<=1
custom_support_vector=X_train[support_vector_mask]
print("\nno of linear SV:",len(custom_support_vector))


#my my mykernal
gamma=0.1 #how quickly similarity decreases with dist
#random features
n_components=100
np.random.seed(42)

w_rff=np.random.normal(0,np.sqrt(2*gamma),(n_features,n_components))
b_rff=np.random.uniform(0,2*np.pi,n_components)

def transform_rff(data):
    return np.sqrt(2.0/n_components)*np.cos(np.dot(data,w_rff)+b_rff)
X_train_rff=transform_rff(X_train)
X_test_rff=transform_rff(X_test)

n_samples_rff,n_features_rff=X_train_rff.shape
W_rff_model=np.zeros(n_features_rff)
B_rff=0.0

for epoch in range(epochs):
    for i in range(n_samples_rff):
        score_rff=np.dot(X_train_rff[i],W_rff_model)+B_rff
        if Y1[i]*score_rff>=1:
            W_rff_model=W_rff_model-alpha*W_rff_model
        else:
            W_rff_model=W_rff_model-alpha*(W_rff_model-c*Y1[i]*X_train_rff[i])
            B_rff=B_rff-alpha*(-c*Y1[i])
y_predd_rff=np.dot(X_test_rff,W_rff_model)+B_rff
y_pred_custom_rff=np.where(y_predd_rff<=0,0,1)
print_metrics(y_test,y_pred_custom_rff,"my kernel svm")

#kernal lib
rbf_model=SVC(kernel="rbf",C=5,gamma="scale")
rbf_model.fit(X_train,y_train)
y_pred_sklearn_rbf=rbf_model.predict(X_test)
print_metrics(y_test,y_pred_sklearn_rbf,"lib svm")

#baseline
mlp_model=MLPClassifier(hidden_layer_sizes=(16,8),activation="relu",
                        max_iter=500,random_state=42)
mlp_model.fit(X_train,y_train)
y_pred_mlp=mlp_model.predict(X_test).flatten()
print_metrics(y_test,y_pred_mlp,"MLP Classifier")